# 03 · Evaluation

Load each best checkpoint and score it once on the held-out test set, saving `metrics.json` + `cm.png` per model.

In [ ]:
import os
from pathlib import Path
# Run from the repo root so the package's relative paths (data/, outputs/) resolve.
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print('cwd:', Path.cwd())

In [ ]:
from malaria_vit.config import MODELS, TrainConfig, SPLIT_CSV, OUTPUTS_DIR
from malaria_vit.data import MalariaDataModule
from malaria_vit.evaluate import evaluate_checkpoint, save_metrics, save_confusion_matrix

for key in MODELS:
    # robust to a versioned filename (best.ckpt / best-v1.ckpt)
    ckpt = next((OUTPUTS_DIR / key / 'ckpts').glob('best*.ckpt'))
    dm = MalariaDataModule(SPLIT_CSV, MODELS[key], TrainConfig())
    m = evaluate_checkpoint(ckpt, dm)
    save_metrics(m, OUTPUTS_DIR / key / 'metrics.json')
    save_confusion_matrix(m['confusion_matrix'], OUTPUTS_DIR / key / 'cm.png', title=key)
    print(key, {k: round(m[k], 4) for k in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']})